# Качество красного вина

Файл `winequality-red.csv` — выборка [Red Wine Quality](https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009): 1599 красных вин «Vinho Verde». Одиннадцать колонок — химический анализ, `quality` — оценка дегустаторов от 3 до 8 в этих данных (шкала 0–10).

Модель отвечает, хорошее ли вино: `quality` от 7 и выше.

In [1]:
import pandas as pd

df = pd.read_csv("winequality-red.csv")
df.shape

(1599, 12)

In [2]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


Признаки — результат лабораторного анализа:

- `fixed acidity`, `volatile acidity`, `citric acid` — фиксированная, летучая и лимонная кислота
- `residual sugar` — остаточный сахар, г/л
- `chlorides` — хлориды
- `free sulfur dioxide`, `total sulfur dioxide` — свободный и общий диоксид серы
- `density` — плотность
- `pH`
- `sulphates` — сульфаты
- `alcohol` — спирт, % об.
- `quality` — оценка дегустаторов

In [3]:
df["quality"].value_counts().sort_index()

quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64

## Целевой класс

`quality` от 7 — хорошее вино (`good` = 1), ниже — обычное (`good` = 0). Исходную оценку в признаки не кладу.

In [4]:
df["good"] = (df["quality"] >= 7).astype(int)
df["good"].value_counts().sort_index().rename({0: "0 — обычное", 1: "1 — хорошее"})

good
0 — обычное    1382
1 — хорошее     217
Name: count, dtype: int64

## Обучающая и тестовая выборки

Деление **80% / 20%**, стратификация по `good`, `random_state=42`.

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["quality", "good"])
y = df["good"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Обучающая: {X_train.shape[0]} строк ({X_train.shape[0] / len(df):.0%})")
print(f"Тестовая:  {X_test.shape[0]} строк ({X_test.shape[0] / len(df):.0%})")

Обучающая: 1279 строк (80%)
Тестовая:  320 строк (20%)


In [6]:
pd.DataFrame(
    {
        "обучение": y_train.value_counts().sort_index(),
        "тест": y_test.value_counts().sort_index(),
        "доля в обучении": y_train.value_counts(normalize=True).sort_index().mul(100).round(1),
        "доля в тесте": y_test.value_counts(normalize=True).sort_index().mul(100).round(1),
    }
).rename(index={0: "0 — обычное", 1: "1 — хорошее"})

,обучение,тест,доля в обучении,доля в тесте
good,,,,
0 — обычное,1105,277,86.4,86.6
1 — хорошее,174,43,13.6,13.4


## Случайный лес

Модель — `RandomForestClassifier` (200 деревьев). Все признаки числовые, пропусков нет, поэтому отдельная подготовка не нужна.

In [7]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

print(f"Точность на обучении: {model.score(X_train, y_train):.1%}")
print(f"Точность на тесте:    {model.score(X_test, y_test):.1%}")

Точность на обучении: 100.0%
Точность на тесте:    94.4%


## Ошибки I и II рода

Положительный класс — хорошее вино (`good` = 1).

- **Ошибка I рода** — обычное вино названо хорошим: FP / (FP + TN).
- **Ошибка II рода** — хорошее вино не распознано: FN / (FN + TP).

In [8]:
from sklearn.metrics import confusion_matrix


def error_summary(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return pd.Series(
        {
            "верно: обычное (TN)": tn,
            "верно: хорошее (TP)": tp,
            "ошибка I рода, вин (FP)": fp,
            "ошибка II рода, вин (FN)": fn,
            "доля ошибок I рода, %": round(100 * fp / (fp + tn), 1),
            "доля ошибок II рода, %": round(100 * fn / (fn + tp), 1),
        }
    )


pd.DataFrame(
    {
        "обучение": error_summary(y_train, model.predict(X_train)),
        "тест": error_summary(y_test, model.predict(X_test)),
    }
)

,обучение,тест
верно: обычное (TN),1105.0,275.0
верно: хорошее (TP),174.0,27.0
"ошибка I рода, вин (FP)",0.0,2.0
"ошибка II рода, вин (FN)",0.0,16.0
"доля ошибок I рода, %",0.0,0.7
"доля ошибок II рода, %",0.0,37.2


## Вывод

На тесте точность **94.4%**. Если всегда отвечать «обычное», она была бы **86.6%**: хороших вин мало, 217 из 1599.

На обучении точность **100%** — лес запомнил обучающую выборку. На тесте обычные вина почти не путает: ошибка I рода **0.7%** (2 из 277). Хорошие пропускает чаще: ошибка II рода **37.2%** (16 из 43). Из хороших вин модель находит **62.8%**.